# Day 2 Submission: Team Priya

**Approach.** We only looked for *people*: records that say `Student S0xx` or `Alumnus A0xx`. We then thought about which other fields might link records and argued against rooms, amounts and semesters, since the same value turns up in unrelated events.

*Mock submission for judging rehearsal. The structural graph verdicts in the outcomes below come from the offline checker; `baseline_fabricated` is `None` because the baseline's wording depends on a live run.*

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os
from openai import OpenAI

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
print("Model + client ready")

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: campus operations & student services

179 records in 14 categories. Four pairs of categories deliberately share vocabulary (money, rooms, passes/permits, events). Most records stand alone; some name a person, a club or a ticket.

In [ ]:
campus_categories = {
    'Admissions': [
        "A new student's admission application was reviewed by the committee",
        'Entrance exam scorecard was verified for a incoming student',
        'Admission offer letter was issued to a selected candidate',
        "A student's transfer application from another college was processed",
        'Document verification was completed for a first-year admission',
        'Admission committee flagged a discrepancy in a submitted transcript',
        'Waitlisted candidate was offered a seat after a dropout',
        'Provisional admission was confirmed pending final marksheet submission',
        "International student's visa documents were reviewed for admission",
        'Admission counselor scheduled a call with a prospective student',
        "Student S013's admission document verification was completed ahead of orientation",
    ],
    'Fees_Finance': [
        'Tuition fee payment of 45000 rupees was received from a student',
        'A late fee penalty was applied to an overdue tuition payment',
        'Fee receipt was generated for the current semester payment',
        'A student requested an installment plan for pending tuition fees',
        "Refund was processed for a cancelled admission's fee payment",
        "Finance office reconciled the semester's total fee collection",
        'A bounced fee payment was flagged for reprocessing',
        'Hostel fee and tuition fee were combined into one invoice',
        "A student's fee payment was adjusted after a course change",
        'Finance team sent a reminder for an unpaid tuition balance',
        "Student S001's tuition fee payment of 42000 rupees was received for the fall semester",
        "Student S002 was charged a late fee penalty on last month's tuition invoice",
        "Student S004's tuition installment plan was approved by the finance office",
        "Student S009's tuition fee payment of 46000 rupees was received for the spring semester",
        "A refund of 46000 rupees was processed for a cancelled admission's fee payment",
    ],
    'Scholarships': [
        'Scholarship disbursement of 15000 rupees was processed for a first-year student',
        'A merit scholarship application was approved by the review panel',
        'Scholarship renewal was submitted for the upcoming academic year',
        "A need-based scholarship award was credited to a student's account",
        'Scholarship committee rejected an incomplete application',
        'Sports scholarship was granted to a state-level athlete',
        "A student's scholarship was revoked due to low attendance",
        'External scholarship provider confirmed funding for a student',
        'Scholarship disbursement was delayed pending document resubmission',
        'A minority scholarship application was forwarded for government approval',
        "Student S001's merit scholarship of 20000 rupees was disbursed for the fall semester",
        "Student S002's scholarship renewal was approved for the second year",
        "Student S003's sports scholarship was confirmed by the athletics department",
    ],
    'Hostel_Accommodation': [
        'Room 214 was allotted to a new student this week',
        'A hostel room change request was approved for a second-year student',
        'Hostel warden resolved a roommate conflict in Block C',
        'A student checked out of the hostel at the end of the semester',
        'New hostel admission form was submitted for the upcoming term',
        "Hostel curfew violation was reported to the warden's office",
        'A student requested a single-occupancy room upgrade',
        'Hostel mess registration was completed for a new resident',
        'Room allotment list was published for incoming hostel students',
        'A student vacated their room early due to a course withdrawal',
        "Student S006's room change request to Room 118 was approved",
        'Student S007 reported a curfew violation warning from the hostel warden',
        'A hostel maintenance ticket (HMT-58) was filed by the student in Room 310 for a broken window',
    ],
    'Facilities_Maintenance': [
        'A maintenance fee payment of 2000 rupees was recorded for Block C rooms',
        'Electrician was dispatched to fix a wiring issue in the hostel block',
        'Plumbing repair request was logged for a leaking bathroom',
        'Facilities team repainted the corridor walls in the academic block',
        'Air conditioning unit was serviced in the seminar hall',
        'A broken window pane was replaced in the library building',
        'Elevator maintenance was scheduled for the administrative block',
        'Facilities flagged a structural crack for engineering inspection',
        'Water cooler was repaired in the student common area',
        'Furniture replacement was approved for the outdated classroom desks',
        'Facilities resolved maintenance ticket HMT-58, replacing the broken window in Room 310',
        "Facilities serviced the air conditioning unit in Room 118 at Student S006's request",
        'Student S008 filed a complaint about a leaking bathroom in a different block',
        'Student S012 reported a broken door lock in Room 118 during the following semester',
        'Plumbing repair in Room 214 was completed after a leak was reported',
    ],
    'Library': [
        'A student issued three reference books for the semester',
        'Overdue library fine was paid by a student for a late return',
        'New journal subscriptions were added to the digital library',
        'A damaged book was reported and replaced in the collection',
        'Library extended its hours during the examination period',
        'A student requested an inter-library loan for a rare textbook',
        'Library staff reshelved returned books after the rush week',
        'E-book access was granted to a student for an online course',
        "A lost book fee was charged to a student's account",
        'Library conducted an annual stock verification of its collection',
        'Student S013 issued two reference textbooks for their first-year courses',
        'Student S015 was charged a lost book fee after misplacing a library textbook',
        'A library fine of 500 rupees was collected from a student for a damaged book',
        'Library extended its seating for the spring semester exam preparation',
    ],
    'IT_Helpdesk': [
        "A student's login credentials were reset after a lockout",
        'Wi-Fi connectivity issue was reported in the academic block',
        'IT helpdesk resolved a printer malfunction in the computer lab',
        "A student's email account was migrated to the new domain",
        'Software installation request was completed for a lab machine',
        'IT flagged a phishing email reported by a faculty member',
        "A student's laptop was unable to connect to the campus network",
        'IT helpdesk ticket was closed after a successful password reset',
        'Server downtime affected the online portal for two hours',
        'A student requested VPN access for remote library resources',
        "Student S014's laptop was unable to connect to the campus Wi-Fi network",
    ],
    'Placement_Careers': [
        'A student was shortlisted for a summer internship interview',
        'Placement cell scheduled a pre-placement talk with a recruiter',
        'A student received a job offer through the campus placement drive',
        'Resume workshop was conducted for final-year students',
        'Placement officer followed up with a company on pending offers',
        'A student withdrew from the placement process after a personal offer',
        'Mock interview session was organized for placement preparation',
        'Company representative visited campus for a recruitment drive',
        "A student's internship certificate was verified by the placement office",
        'Placement statistics were compiled for the annual report',
        'Student S013 was shortlisted for a summer internship interview by a recruiting company',
    ],
    'Events_Clubs': [
        'The robotics club organized a workshop for new members',
        'Cultural fest committee finalized the event schedule',
        'A student club requested funding for an upcoming competition',
        'Annual sports day events were scheduled across three days',
        'The photography club held an exhibition in the main hall',
        "Student council approved a new club's registration",
        'A guest lecture was organized by the entrepreneurship cell',
        'Debate club members prepared for an inter-college competition',
        "The music club performed at the college's annual day",
        "A club's event proposal was rejected due to a venue conflict",
        "The robotics club's workshop for new members was led by its club president (Club RC-1)",
        "The robotics club's president also represented the club (Club RC-1) at the inter-college competition",
    ],
    'Transport': [
        'A new bus route was added for students from the north campus',
        'Transport pass renewal was completed for the semester',
        'A student reported a delay on the college shuttle service',
        'Transport office adjusted the bus schedule for exam week',
        'A damaged bus seat was reported to the transport department',
        'Student transport fee was included in the semester invoice',
        'A new driver was assigned to the evening shuttle route',
        'Transport helpdesk resolved a lost bus pass complaint',
        'Additional buses were arranged for the college fest weekend',
        'A student requested a refund for an unused transport pass',
        "Student S009's bus pass was renewed for the spring semester",
        'Student S011 filed a complaint about a delayed shuttle on the north campus route',
        'Transport office added extra shuttle trips for the spring semester',
    ],
    'Parking_Permits': [
        "A student's parking permit application was approved for the fall semester",
        'Parking enforcement issued a warning for an expired permit',
        'A visitor parking pass was requested for a campus event',
        'Parking permit renewal was completed for a returning student',
        'A student reported a lost parking permit sticker',
        'Parking office resolved a dispute over an assigned spot',
        'A faculty parking permit was upgraded to a reserved spot',
        'Parking permit fee was waived for a shared carpool application',
        'A damaged permit sticker was replaced at the parking office',
        "Student S016's parking permit application was approved for the spring term",
        'Parking violation notice was issued for an unregistered vehicle',
        "Student S009's parking permit was renewed for the spring semester",
        'Student S010 received a parking violation notice for an unregistered vehicle',
        'A parking fine of 500 rupees was collected from a student for an unregistered vehicle',
    ],
    'Alumni_Relations': [
        'An alumni donation was received for the new library wing',
        'Alumni association organized a networking event for graduates',
        "A former student's contact details were updated in the alumni database",
        'Alumni newsletter was sent out for the quarterly update',
        'An alumni-funded scholarship endowment was reviewed by the committee',
        'Alumni relations office confirmed a guest speaker for homecoming',
        'A batch reunion event was scheduled for December',
        'Alumni association approved funding for a new mentorship program',
        'A distinguished alumnus was invited to the annual convocation',
        'Alumni donation records were reconciled for the fiscal year',
        "Alumnus A001's donation for the new library wing was formally acknowledged by the college",
        'Alumnus A001 attended the homecoming networking event hosted by the alumni association',
        "Alumnus A002's mentorship program funding was approved by the alumni association",
        'Alumnus A002 was invited to speak at the annual convocation as a distinguished alumnus',
    ],
    'Exam_Results': [
        "A student's semester grade report was published on the portal",
        'Exam committee resolved a discrepancy in a marksheet',
        'A student requested a re-evaluation of their exam paper',
        'Final exam results were released for the engineering department',
        "A student's backlog exam was scheduled for the next session",
        'Exam office issued a duplicate marksheet for a lost original',
        'A grade correction was approved after a clerical error was found',
        'Semester GPA was recalculated after a course withdrawal',
        'Exam hall ticket was issued for the upcoming semester exams',
        "A student's provisional degree certificate was requested after final results",
        "Student S014's backlog exam for a failed course was scheduled for next session",
    ],
    'Campus_Security': [
        'Campus security responded to a report of a stolen bicycle',
        'A lost ID card was recovered and returned through campus security',
        'Security patrol flagged an unlocked door in the academic block',
        'A student reported a suspicious person near the hostel gate',
        'Campus security reviewed CCTV footage after a minor incident',
        'A found wallet was logged at the campus security office',
        'Security escorted a student to their vehicle after a late-night event',
        'A broken security gate was reported for repair',
        'Campus security issued a visitor pass for an external vendor',
        'An access card malfunction was resolved by the security office',
        'Student S015 reported a lost ID card to campus security',
        "Student S016's vehicle was found unlocked by a security patrol and secured",
    ],
}

campus_texts = []
campus_labels = []
for category, sentences in campus_categories.items():
    for s in sentences:
        campus_texts.append(s)
        campus_labels.append(category)

print(f"{len(campus_texts)} records, {len(campus_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
campus_embeddings = model.encode(campus_texts, show_progress_bar=True)
campus_embeddings = np.ascontiguousarray(campus_embeddings.astype('float32'))

dim = campus_embeddings.shape[1]
campus_index = hnswlib.Index(space='cosine', dim=dim)
campus_index.init_index(max_elements=len(campus_texts), ef_construction=200, M=16)
campus_index.add_items(campus_embeddings, np.arange(len(campus_texts)))
campus_index.set_ef(50)
print(f"Index built: {len(campus_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = campus_index.knn_query(q_emb, k=k)
    return [(int(idx), campus_texts[idx], campus_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    response = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return {"question": question, "retrieved": results, "answer": response.choices[0].message.content}

## 5. Explore

In [ ]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(campus_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

## 6. Our pattern registry

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Student',
        "regex": r"\bStudent\s+(S\d{3})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Student S001',
        "why": 'A student ID names one specific person: two records naming S001 are about the same student.',
    },
    {
        "name": 'Alumnus',
        "regex": r"\bAlumnus\s+(A\d{3})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Alumnus A001',
        "why": 'An alumnus ID names one person, exactly like a student ID: the donation and homecoming records for A001 describe the same alumnus.',
    },
]

REJECTED_CANDIDATES = [
    {"candidate": 'Room numbers', "kind": 2, "why_rejected": 'A room is a place used by many different students over time (Room 118 is mentioned for two different students), so it does not identify one event.'},
    {"candidate": 'Amounts', "kind": 2, "why_rejected": 'Two payments can have the same amount without being related.'},
    {"candidate": 'Semester', "kind": 2, "why_rejected": 'Far too many records share a semester.'},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 7. Build the graph from our registry

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(campus_texts, campus_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(campus_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 8. Grounded answer (given)

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about campus records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Amounts, room numbers, block names, semesters, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    response = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": response.choices[0].message.content}

## 9. Public question pack

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'kind': 'Vocabulary overlap (money)',
  'question': 'When the scholarship was applied to the tuition fee, how much of the fee was left '
              'to pay?',
  'trap_records': ["A need-based scholarship award was credited to a student's account",
                   'Finance team sent a reminder for an unpaid tuition balance'],
  'correct_answer_must_state': 'Nothing links this scholarship record to that unpaid balance, so '
                               'the remaining fee cannot be worked out from the data.',
  'expect_connected': False},
 {'id': 'P2',
  'kind': 'Vocabulary overlap (room)',
  'question': 'Did the student who vacated their room early leave because of the leaking bathroom?',
  'trap_records': ['A student vacated their room early due to a course withdrawal',
                   'Plumbing repair request was logged for a leaking bathroom'],
  'correct_answer_must_state': 'No stated connection: the room vacated and the bathroom leak are '
                               'separate records and name no shared student.',
  'expect_connected': False},
 {'id': 'P3',
  'kind': 'Vocabulary overlap (pass/permit)',
  'question': "Was the returning student's parking permit renewed together with their transport "
              'pass?',
  'trap_records': ['Transport pass renewal was completed for the semester',
                   'Parking permit renewal was completed for a returning student'],
  'correct_answer_must_state': 'No evidence these two renewals belong to the same student; they '
                               'are separate records.',
  'expect_connected': False},
 {'id': 'P4',
  'kind': 'Vocabulary overlap (event)',
  'question': 'Was the alumni networking event part of the cultural fest schedule?',
  'trap_records': ['Alumni association organized a networking event for graduates',
                   'Cultural fest committee finalized the event schedule'],
  'correct_answer_must_state': 'No stated connection: both mention an event, but nothing ties them '
                               'to the same event.',
  'expect_connected': False},
 {'id': 'P5',
  'kind': 'Kind 2 trap: room number',
  'question': 'Did the new student allotted Room 214 cause the plumbing leak in that room?',
  'trap_records': ['Room 214 was allotted to a new student this week',
                   'Plumbing repair in Room 214 was completed after a leak was reported'],
  'correct_answer_must_state': 'No stated connection: both mention Room 214, but a room number is '
                               'a location reused over time, not an event identifier.',
  'expect_connected': False},
 {'id': 'P6',
  'kind': 'Kind 2 trap: block',
  'question': 'Did the roommate conflict in Block C lead to the Block C maintenance fee?',
  'trap_records': ['Hostel warden resolved a roommate conflict in Block C',
                   'A maintenance fee payment of 2000 rupees was recorded for Block C rooms'],
  'correct_answer_must_state': 'No stated connection: sharing a block name is not sharing an '
                               'event.',
  'expect_connected': False},
 {'id': 'P7',
  'kind': 'Kind 2 trap: amount',
  'question': 'Was the 500 rupee library fine paid to cover the parking fine?',
  'trap_records': ['A library fine of 500 rupees was collected from a student for a damaged book',
                   'A parking fine of 500 rupees was collected from a student for an unregistered '
                   'vehicle'],
  'correct_answer_must_state': 'No stated connection: the two fines happen to be the same amount, '
                               'which is a coincidence of value, not a shared event.',
  'expect_connected': False},
 {'id': 'P8',
  'kind': 'Cross-category causal',
  'question': 'Did the server downtime affect the pending placement offers?',
  'trap_records': ['Server downtime affected the online portal for two hours',
                   'Placement officer followed up with a company on pending offers'],
  'correct_answer_must_state': 'No stated connection between an IT outage and a placement '
                               'follow-up; any causal story is invented.',
  'expect_connected': False}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Student S002 charged a late fee and also approved for scholarship renewal?',
  'shared_identifier': 'Student S002',
  'shared_records': ["Student S002 was charged a late fee penalty on last month's tuition invoice",
                     "Student S002's scholarship renewal was approved for the second year"],
  'correct_answer_must_state': 'Yes -- both records name Student S002.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Did Student S009 renew both their bus pass and their parking permit?',
  'shared_identifier': 'Student S009',
  'shared_records': ["Student S009's bus pass was renewed for the spring semester",
                     "Student S009's parking permit was renewed for the spring semester"],
  'correct_answer_must_state': 'Yes -- both records name Student S009.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': "Was Alumnus A002's mentorship funding approved, and was the same alumnus invited to "
              'speak at convocation?',
  'shared_identifier': 'Alumnus A002',
  'shared_records': ["Alumnus A002's mentorship program funding was approved by the alumni "
                     'association',
                     'Alumnus A002 was invited to speak at the annual convocation as a '
                     'distinguished alumnus'],
  'correct_answer_must_state': 'Yes -- both records name Alumnus A002.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{campus_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

In [ ]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)
pos_rows = run_pack(PUBLIC_POSITIVE)

## 10. Outcomes and the failure we still have

In [ ]:
OUTCOMES = [{'id': 'P1',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected: no person is named in either record.'},
 {'id': 'P2',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected: no shared person.'},
 {'id': 'P3',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected: no shared person.'},
 {'id': 'P4',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected: no shared person.'},
 {'id': 'P5',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected. Both mention Room 214, which we rejected as a link: it is a '
                 'place.'},
 {'id': 'P6',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': "NOT CONNECTED as expected: no pattern covers 'Block C'."},
 {'id': 'P7',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected. The same 500-rupee amount is a coincidence; we rejected '
                 'amounts.'},
 {'id': 'P8',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'NOT CONNECTED as expected.'},
 {'id': 'PP1',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'CONNECTED via Student S002.'},
 {'id': 'PP2',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'CONNECTED via Student S009.'},
 {'id': 'PP3',
  'baseline_fabricated': None,
  'grounded_correct': True,
  'explanation': 'CONNECTED via Alumnus A002, thanks to the Alumnus pattern.'}]

REMAINING_FAILURE = "We only searched for people. We did not check whether other ID-like strings in the data (anything that is not a Student or Alumnus ID) identify events. If they do, we would answer 'no recorded connection' to questions whose true answer is yes."

## 11. Submission summary

In [ ]:
def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()